# 06 — Yoko Musanga × Google Satellite Embedding regression

## Question
Can the **mean UAV-derived Musanga probability at 10 m** be predicted from the 64-dimensional Google Satellite Embedding for the Yoko study site?

This is deliberately a **single-site proof of concept**. It uses a random train/test split for rapid model development. Cross-site validation should be added later when the other UAV Musanga probability products are introduced.

## Workflow
1. Load the validated UAV `musanga_probability_uint16.tif`.
2. Download the **2020 Google Satellite Embedding V1** over the UAV footprint.
3. Use the embedding raster as the authoritative satellite grid.
4. Aggregate UAV probability onto that grid using **average resampling** — no Musanga threshold.
5. Build a table containing `musanga_prob_mean` and A00…A63.
6. Benchmark Random Forest and HistGradientBoosting regressors.
7. Export observed, predicted, and residual 10-m GeoTIFFs.

The target remains continuous throughout.

## BLOCK 01 — Install and import packages

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 01 — INSTALL AND IMPORT PACKAGES")
print("=" * 100)

!pip -q install earthengine-api geemap rasterio pyproj scikit-learn scipy joblib

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.stats import spearmanr
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import ee
import geemap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
print("Imports OK")

## BLOCK 02 — Mount Google Drive and define paths

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 02 — MOUNT DRIVE AND DEFINE PATHS")
print("=" * 100)

from google.colab import drive
DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

PROJECT_ROOT = MYDRIVE / "Colab Notebooks" / "Musanga"
WORK_DIR = PROJECT_ROOT / "work"
INPUT_PROB = WORK_DIR / "whole_orthomosaic_inference" / "musanga_probability_uint16.tif"

OUT_DIR = WORK_DIR / "yoko_satellite_embedding"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_TIF = OUT_DIR / "Yoko_GoogleSatelliteEmbedding_2020_10m.tif"
OBS_TIF = OUT_DIR / "Yoko_UAV_Musanga_mean_10m.tif"
PRED_TIF = OUT_DIR / "Yoko_Embedding_Musanga_predicted_10m.tif"
RESID_TIF = OUT_DIR / "Yoko_Embedding_Musanga_residual_10m.tif"
TABLE_CSV = OUT_DIR / "Yoko_embedding_training_table.csv"
MODEL_FILE = OUT_DIR / "Yoko_best_embedding_regressor.joblib"
SUMMARY_JSON = OUT_DIR / "Yoko_embedding_regression_summary.json"

assert INPUT_PROB.exists(), f"Missing input: {INPUT_PROB}"
print("INPUT:", INPUT_PROB)
print("OUTPUT:", OUT_DIR)

## BLOCK 03 — Configuration and inspect UAV probability raster

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 03 — CONFIGURATION / UAV INPUT")
print("=" * 100)

EE_PROJECT = "tropsedslu"
EMBEDDING_YEAR = 2020
TEST_SIZE = 0.20
FORCE_DOWNLOAD_EMBEDDINGS = False
FORCE_REFIT_MODELS = False
MODEL_CACHE_DIR = OUT_DIR / "model_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Model cache:", MODEL_CACHE_DIR)

with rasterio.open(INPUT_PROB) as src:
    print("CRS:", src.crs)
    print("Shape:", src.height, src.width)
    print("Resolution:", src.res)
    print("dtype:", src.dtypes[0], "nodata:", src.nodata)
    print("bounds:", src.bounds)

    # Sparse sample only: do not load the full UAV raster.
    oh = min(1000, src.height)
    ow = min(1000, src.width)
    sample = src.read(1, out_shape=(oh, ow), resampling=Resampling.nearest).astype("float32")
    if src.nodata is not None:
        sample = sample[sample != src.nodata]
    sample = sample[np.isfinite(sample)]

print("Sample min/max:", float(sample.min()), float(sample.max()))
# Authoritative encoding of musanga_probability_uint16.tif:
# stored uint16 value * 0.0001 = probability.
PROB_SCALE = 10000.0
print("Probability divisor:", PROB_SCALE)
print("Embedding year:", EMBEDDING_YEAR)

## BLOCK 04 — Authenticate Earth Engine and download the 2020 embedding

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 04 — EARTH ENGINE / GOOGLE SATELLITE EMBEDDING")
print("=" * 100)

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

with rasterio.open(INPUT_PROB) as src:
    b = src.bounds
    left, bottom, right, top = transform_bounds(
        src.crs, "EPSG:4326", b.left, b.bottom, b.right, b.top, densify_pts=21
    )

roi = ee.Geometry.Rectangle([left, bottom, right, top], geodesic=False)

collection = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f"{EMBEDDING_YEAR}-01-01", f"{EMBEDDING_YEAR + 1}-01-01")
    .filterBounds(roi)
)
print("Embedding images intersecting Yoko:", collection.size().getInfo())

band_names = [f"A{i:02d}" for i in range(64)]
embedding = collection.mosaic().select(band_names)

# Backward compatibility: move/copy an already-downloaded embedding from the old folder.
OLD_EMBED_TIF = OUT_DIR / "Yoko_GoogleSatelliteEmbedding_2020_10m.tif"
if (not EMBED_TIF.exists()) and OLD_EMBED_TIF.exists():
    import shutil
    shutil.copy2(OLD_EMBED_TIF, EMBED_TIF)
    print("Copied existing embedding into persistent sat cache:", EMBED_TIF)

if FORCE_DOWNLOAD_EMBEDDINGS or not EMBED_TIF.exists():
    print("Downloading 64-band embedding raster...")
    geemap.ee_export_image(
        embedding,
        filename=str(EMBED_TIF),
        scale=10,
        region=roi,
        file_per_band=False,
    )
else:
    print("Reusing:", EMBED_TIF)

assert EMBED_TIF.exists(), "Embedding export failed."
with rasterio.open(EMBED_TIF) as src:
    print("Embedding CRS:", src.crs)
    print("Embedding shape:", src.height, src.width)
    print("Embedding bands:", src.count)
    print("Embedding resolution:", src.res)
    assert src.count == 64, f"Expected 64 bands, got {src.count}"

## BLOCK 05 — Aggregate UAV probability directly to the embedding grid

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 05 — UAV MEAN PROBABILITY ON EMBEDDING GRID (PERSISTENT CACHE)")
print("=" * 100)

with rasterio.open(EMBED_TIF) as emb:
    dst_crs = emb.crs
    dst_transform = emb.transform
    dst_shape = (emb.height, emb.width)
    out_profile = emb.profile.copy()

# Reuse the expensive aggregation whenever it is already on Drive.
if OBS_TIF.exists():
    print("Reusing cached aggregation:", OBS_TIF)
    with rasterio.open(OBS_TIF) as src:
        cached = src.read(1).astype("float32")
        cached_nodata = src.nodata
    good = np.isfinite(cached)
    if cached_nodata is not None:
        good &= cached != cached_nodata
    obs = np.where(good, cached, np.nan).astype("float32")
    source_nodata = 65535
else:
    print("No cached aggregation found — computing once and writing to:", OBS_TIF)
    with rasterio.open(INPUT_PROB) as src:
        source_nodata = src.nodata
        if source_nodata is None:
            raise ValueError("UAV probability raster has no declared NoData value; cannot safely aggregate.")
        obs_raw = np.full(dst_shape, np.nan, dtype="float32")
        reproject(
            source=rasterio.band(src, 1),
            destination=obs_raw,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=source_nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.average,
            init_dest_nodata=True,
        )

    # Authoritative uint16 encoding: stored value * 0.0001 = probability.
    obs = obs_raw / PROB_SCALE
    good = np.isfinite(obs) & (obs >= 0.0) & (obs <= 1.0)

    profile1 = out_profile.copy()
    profile1.update(count=1, dtype="float32", nodata=-9999.0, compress="deflate")
    with rasterio.open(OBS_TIF, "w", **profile1) as dst:
        dst.write(np.where(good, obs, -9999.0).astype("float32"), 1)
    print("Cached aggregation saved.")

# Define export profile also when loading from cache.
profile1 = out_profile.copy()
profile1.update(count=1, dtype="float32", nodata=-9999.0, compress="deflate")

print("Source UAV NoData value:", source_nodata)
print("Satellite-grid cells:", obs.size)
print("Valid UAV target cells:", int(good.sum()))
print("Excluded / NoData cells:", int((~good).sum()))
print(f"Valid fraction of embedding grid: {good.mean():.3%}")
print("Mean target:", float(np.nanmean(obs[good])))
print("Median target:", float(np.nanmedian(obs[good])))
print("P90 target:", float(np.nanpercentile(obs[good], 90)))
print("Max target:", float(np.nanmax(obs[good])))
print("Persistent target:", OBS_TIF)

## BLOCK 06 — Inspect target distribution

## BLOCK 05B — Visual QA: raw UAV, 10 m aggregation, satellite imagery and Earth Engine map\n\nThis is a visual check only. It does not alter the modelling target. The satellite panel uses a cloud-filtered Sentinel-2 RGB composite close to the UAV survey (24–25 August 2020).\n

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 05B — VISUAL QA / UAV → 10 m → SATELLITE")
print("=" * 100)

# 1) Raw UAV probability, downsampled only for display (not for modelling).
with rasterio.open(INPUT_PROB) as src:
    dh = min(1800, src.height)
    dw = max(1, int(src.width * dh / src.height))
    raw = src.read(1, out_shape=(dh, dw), resampling=Resampling.average).astype("float32")
    raw_nodata = src.nodata
    raw = raw / PROB_SCALE
    if raw_nodata is not None:
        # The display resampling can mix edge NoData; use source mask for display validity.
        m = src.dataset_mask(out_shape=(dh, dw), resampling=Resampling.nearest) > 0
        raw[~m] = np.nan
    raw[(raw < 0) | (raw > 1)] = np.nan

# 2) Cached 10-m target.
agg_show = np.where(good, obs, np.nan)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(raw, vmin=0, vmax=1)
ax.set_title("Raw UAV Musanga probability — display overview")
ax.axis("off")
plt.colorbar(im, ax=ax, label="Musanga probability")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(agg_show, vmin=0, vmax=1)
ax.set_title("UAV Musanga probability aggregated to Google embedding grid (~10 m)")
ax.axis("off")
plt.colorbar(im, ax=ax, label="Mean UAV Musanga probability")
plt.tight_layout()
plt.show()

# 3) Sentinel-2 RGB around the UAV survey dates (24–25 Aug 2020).
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2020-08-01", "2020-09-16")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
)
print("Sentinel-2 candidate scenes (2020-08-01 to 2020-09-15):", s2.size().getInfo())
s2_rgb = s2.median().select(["B4", "B3", "B2"])

# Interactive Earth Engine map: Sentinel-2 plus semi-transparent aggregated UAV target.
m = geemap.Map()
m.centerObject(roi, 15)
m.addLayer(s2_rgb, {"min": 0, "max": 3000, "gamma": 1.2}, "Sentinel-2 RGB Aug/Sep 2020")

# Uploading the local 10-m raster back to EE is unnecessary for QA.
# Show the UAV footprint on the EE map and use the two local plots above for exact target alignment.
m.addLayer(roi, {"color": "yellow"}, "UAV / embedding ROI")
display(m)

print("QA note: satellite imagery is a cloud-filtered median near the UAV survey dates,")
print("while Google Satellite Embedding V1 is annual (2020), not a single-month image.")

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 06 — TARGET DISTRIBUTION")
print("=" * 100)

y_all = obs[good]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_all, bins=50)
axes[0].set_xlabel("Mean UAV Musanga probability at 10 m")
axes[0].set_ylabel("Number of cells")
axes[0].set_title("Full distribution")

positive_view = y_all[y_all > 0.001]
if len(positive_view):
    axes[1].hist(positive_view, bins=50)
axes[1].set_xlabel("Mean UAV Musanga probability")
axes[1].set_title("Cells > 0.001 (display only)")
plt.tight_layout()
plt.show()

for q in [0, .25, .5, .75, .9, .95, .99, 1]:
    print(f"q{q:>4}: {np.quantile(y_all, q):.6f}")

## BLOCK 07 — Build canonical 10-m training table

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 07 — BUILD TRAINING TABLE")
print("=" * 100)

with rasterio.open(EMBED_TIF) as src:
    Xcube = src.read().astype("float32")
    emb_nodata = src.nodata
    transform = src.transform

emb_valid = np.all(np.isfinite(Xcube), axis=0)
if emb_nodata is not None:
    emb_valid &= np.all(Xcube != emb_nodata, axis=0)

mask = good & emb_valid
rows, cols = np.where(mask)
X = Xcube[:, mask].T
target = obs[mask]
xs, ys = rasterio.transform.xy(transform, rows, cols, offset="center")
df = pd.DataFrame({
    "cell_id": np.arange(len(target), dtype=int),
    "x": np.asarray(xs),
    "y": np.asarray(ys),
    "musanga_prob_mean": target,
})
for i in range(64):
    df[f"A{i:02d}"] = X[:, i]

df.to_csv(TABLE_CSV, index=False)
print("Rows:", len(df))
print("Only cells with a finite UAV target AND all 64 finite embedding bands are included.")
print("NoData cells are excluded, not assigned target=0.")
print("Predictors:", X.shape[1])
print("Saved:", TABLE_CSV)
display(df.head())

## BLOCK 08 — Random Yoko train/test benchmark

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08 — RANDOM TRAIN/TEST BENCHMARK")
print("=" * 100)

idx = np.arange(len(target))
train_idx, test_idx = train_test_split(
    idx, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, max_features="sqrt",
        n_jobs=-1, random_state=RANDOM_SEED
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.05, l2_regularization=1.0,
        random_state=RANDOM_SEED
    ),
}

results, fitted = [], {}
for name, model in models.items():
    model.fit(X[train_idx], target[train_idx])
    pred = np.clip(model.predict(X[test_idx]), 0, 1)
    results.append({
        "model": name,
        "RMSE": mean_squared_error(target[test_idx], pred) ** 0.5,
        "MAE": mean_absolute_error(target[test_idx], pred),
        "R2": r2_score(target[test_idx], pred),
        "Spearman_rho": spearmanr(target[test_idx], pred).statistic,
    })
    fitted[name] = model

baseline = np.full(len(test_idx), target[train_idx].mean())
results.append({
    "model": "Mean_baseline",
    "RMSE": mean_squared_error(target[test_idx], baseline) ** 0.5,
    "MAE": mean_absolute_error(target[test_idx], baseline),
    "R2": r2_score(target[test_idx], baseline),
    "Spearman_rho": np.nan,
})

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

candidate_results = results_df[results_df["model"] != "Mean_baseline"]
best_name = candidate_results.iloc[0]["model"]
best_model = fitted[best_name]
print("Selected model:", best_name)

# Keep the held-out predictions for the diagnostic block.
test_observed = target[test_idx].copy()
test_predictions = {
    name: np.clip(model.predict(X[test_idx]), 0, 1)
    for name, model in fitted.items()
}
best_test_pred = test_predictions[best_name]


## BLOCK 08B — Abundance-stratified held-out diagnostic\n\nEvaluate M01 separately across observed UAV Musanga abundance classes. Bins are descriptive only and do not alter model training or the continuous target.\n

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08B — ABUNDANCE-STRATIFIED HELD-OUT DIAGNOSTIC")
print("=" * 100)

y_true = test_observed
y_pred = best_test_pred

# Ecologically interpretable descriptive bins; model remains continuous.
edges = np.array([0.0, 0.01, 0.05, 0.20, 0.50, 1.000001])
labels = ["0–0.01", "0.01–0.05", "0.05–0.20", "0.20–0.50", ">0.50"]

bin_id = np.digitize(y_true, edges[1:-1], right=False)
diag_rows = []
for b, label in enumerate(labels):
    m = bin_id == b
    n = int(m.sum())
    if n == 0:
        continue
    yt, yp = y_true[m], y_pred[m]
    diag_rows.append({
        "observed_bin": label,
        "n": n,
        "fraction_test": n / len(y_true),
        "observed_mean": float(np.mean(yt)),
        "predicted_mean": float(np.mean(yp)),
        "bias_pred_minus_obs": float(np.mean(yp - yt)),
        "MAE": float(mean_absolute_error(yt, yp)),
        "RMSE": float(mean_squared_error(yt, yp) ** 0.5),
        "Spearman_rho": float(spearmanr(yt, yp).statistic) if n >= 3 else np.nan,
    })

abundance_diag = pd.DataFrame(diag_rows)
display(abundance_diag)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_true, y_pred, s=10, alpha=0.35)
ax.plot([0, 1], [0, 1], "--", linewidth=1)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Observed mean UAV Musanga probability")
ax.set_ylabel("Predicted probability")
ax.set_title(f"M01 held-out predictions — {best_name}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
xpos = np.arange(len(abundance_diag))
ax.plot(xpos, abundance_diag["observed_mean"], marker="o", label="Observed mean")
ax.plot(xpos, abundance_diag["predicted_mean"], marker="o", label="Predicted mean")
ax.set_xticks(xpos)
ax.set_xticklabels(abundance_diag["observed_bin"], rotation=30, ha="right")
ax.set_ylabel("Mean Musanga probability")
ax.set_xlabel("Observed abundance class")
ax.set_title("M01 response across Musanga abundance")
ax.legend()
plt.tight_layout()
plt.show()

print("\nInterpretation aid:")
print("- predicted_mean << observed_mean in high bins = underestimation of Musanga-rich cells")
print("- predicted_mean >> observed_mean in low bins = false-positive / smoothing tendency")
print("- Spearman shows within-bin ranking skill; n should be considered for rare high bins")


## BLOCK 08C — M02 abundance-weighted embedding regression

M02 keeps the **same 64 Google Satellite Embedding predictors and the identical held-out test cells as M01**, but weights the training loss so rare Musanga-rich cells contribute more strongly. The continuous target is unchanged and the test set is never weighted.

Weights are inverse-frequency by the same five abundance bins used in Block 08B, normalized to mean 1 and capped to avoid letting the rarest bin dominate. Random Forest is used because it was the best M01 architecture.

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08C — M02 ABUNDANCE-WEIGHTED EMBEDDING REGRESSION")
print("=" * 100)

# Same abundance classes as Block 08B. These bins define TRAINING WEIGHTS only;
# the regression target itself remains continuous.
weight_edges = np.array([0.0, 0.01, 0.05, 0.20, 0.50, 1.000001])
weight_labels = ["0–0.01", "0.01–0.05", "0.05–0.20", "0.20–0.50", ">0.50"]
y_train = target[train_idx]
train_bin = np.digitize(y_train, weight_edges[1:-1], right=False)

# Inverse-frequency weighting gives each abundance class approximately equal
# total influence before capping. Normalize to mean 1 for interpretability.
counts = np.bincount(train_bin, minlength=len(weight_labels))
raw_bin_weights = len(y_train) / (len(weight_labels) * np.maximum(counts, 1))
sample_weight = raw_bin_weights[train_bin]
sample_weight /= sample_weight.mean()

# Cap extreme weights, then renormalize. This is intentionally conservative.
WEIGHT_CAP = 10.0
sample_weight = np.minimum(sample_weight, WEIGHT_CAP)
sample_weight /= sample_weight.mean()

weight_table = pd.DataFrame({
    "observed_bin": weight_labels,
    "n_train": counts,
    "fraction_train": counts / len(y_train),
    "raw_inverse_frequency_weight": raw_bin_weights,
})
weight_table["effective_mean_weight"] = [
    float(sample_weight[train_bin == b].mean()) if counts[b] else np.nan
    for b in range(len(weight_labels))
]
display(weight_table)
print(f"Sample-weight range: {sample_weight.min():.3f} – {sample_weight.max():.3f}")
print(f"Mean sample weight: {sample_weight.mean():.3f}")

# M02 = same RF architecture as M01, only abundance-weighted fitting differs.
m02_model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
m02_model.fit(X[train_idx], y_train, sample_weight=sample_weight)
m02_pred = np.clip(m02_model.predict(X[test_idx]), 0, 1)

def regression_metrics(y, p):
    return {
        "RMSE": float(mean_squared_error(y, p) ** 0.5),
        "MAE": float(mean_absolute_error(y, p)),
        "R2": float(r2_score(y, p)),
        "Spearman_rho": float(spearmanr(y, p).statistic),
    }

m01_pred = best_test_pred
m01_metrics = regression_metrics(test_observed, m01_pred)
m02_metrics = regression_metrics(test_observed, m02_pred)

m01_m02_overall = pd.DataFrame([
    {"model": f"M01 unweighted {best_name}", **m01_metrics},
    {"model": "M02 abundance-weighted RandomForest", **m02_metrics},
])
print("\nOverall held-out comparison — identical test cells:")
display(m01_m02_overall)

# Identical observed bins for a direct M01 versus M02 comparison.
test_bin = np.digitize(test_observed, weight_edges[1:-1], right=False)
comparison_rows = []
for b, label in enumerate(weight_labels):
    m = test_bin == b
    n = int(m.sum())
    if n == 0:
        continue
    yt = test_observed[m]
    p1 = m01_pred[m]
    p2 = m02_pred[m]
    comparison_rows.append({
        "observed_bin": label,
        "n_test": n,
        "observed_mean": float(yt.mean()),
        "M01_pred_mean": float(p1.mean()),
        "M02_pred_mean": float(p2.mean()),
        "M01_bias": float(np.mean(p1 - yt)),
        "M02_bias": float(np.mean(p2 - yt)),
        "M01_MAE": float(mean_absolute_error(yt, p1)),
        "M02_MAE": float(mean_absolute_error(yt, p2)),
        "M01_RMSE": float(mean_squared_error(yt, p1) ** 0.5),
        "M02_RMSE": float(mean_squared_error(yt, p2) ** 0.5),
        "M01_Spearman": float(spearmanr(yt, p1).statistic) if n >= 3 else np.nan,
        "M02_Spearman": float(spearmanr(yt, p2).statistic) if n >= 3 else np.nan,
    })

m01_m02_by_abundance = pd.DataFrame(comparison_rows)
print("\nAbundance-stratified held-out comparison:")
display(m01_m02_by_abundance)

fig, ax = plt.subplots(figsize=(8, 4))
xpos = np.arange(len(m01_m02_by_abundance))
ax.plot(xpos, m01_m02_by_abundance["observed_mean"], marker="o", label="Observed")
ax.plot(xpos, m01_m02_by_abundance["M01_pred_mean"], marker="o", label="M01 unweighted")
ax.plot(xpos, m01_m02_by_abundance["M02_pred_mean"], marker="o", label="M02 weighted")
ax.set_xticks(xpos)
ax.set_xticklabels(m01_m02_by_abundance["observed_bin"], rotation=30, ha="right")
ax.set_xlabel("Observed Musanga abundance class")
ax.set_ylabel("Mean Musanga probability")
ax.set_title("M01 vs M02 — identical held-out cells")
ax.legend()
plt.tight_layout()
plt.show()

print("\nM02 interpretation:")
print("- Improvement in the high-abundance bins indicates that class imbalance was suppressing the M01 upper tail.")
print("- A large increase in the 0–0.01 predicted mean indicates the weighting is buying sensitivity at the cost of false-positive inflation.")
print("- Overall RMSE may worsen even when the Musanga-rich tail improves; judge both the overall and stratified results.")


## BLOCK 08D — M03 moderate abundance-weighted embedding regression

M03 keeps the **same Random Forest, predictors, train/test split, and continuous target** as M01/M02, but uses the square root of the inverse-frequency abundance weights. This is a deliberately milder weighting experiment: it tests whether some upper-tail recovery can be retained without the low-abundance inflation seen in M02.


In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08D — M03 MODERATE ABUNDANCE-WEIGHTED EMBEDDING REGRESSION")
print("=" * 100)

# Reuse the exact M02 training bins/counts and identical M01/M02 held-out cells.
# sqrt(inverse-frequency) moderates the ~40x M02 weight range.
m03_bin_weights = np.sqrt(raw_bin_weights)
m03_sample_weight = m03_bin_weights[train_bin]
m03_sample_weight /= m03_sample_weight.mean()

m03_weight_table = pd.DataFrame({
    "observed_bin": weight_labels,
    "n_train": counts,
    "fraction_train": counts / len(y_train),
    "sqrt_inverse_frequency_weight_raw": m03_bin_weights,
})
m03_weight_table["effective_mean_weight"] = [
    float(m03_sample_weight[train_bin == b].mean()) if counts[b] else np.nan
    for b in range(len(weight_labels))
]
display(m03_weight_table)
print(f"Sample-weight range: {m03_sample_weight.min():.3f} – {m03_sample_weight.max():.3f}")
print(f"Mean sample weight: {m03_sample_weight.mean():.3f}")

m03_model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
m03_model.fit(X[train_idx], y_train, sample_weight=m03_sample_weight)
m03_pred = np.clip(m03_model.predict(X[test_idx]), 0, 1)
m03_metrics = regression_metrics(test_observed, m03_pred)

m01_m02_m03_overall = pd.DataFrame([
    {"model": f"M01 unweighted {best_name}", **m01_metrics},
    {"model": "M02 inverse-frequency weighted RF", **m02_metrics},
    {"model": "M03 sqrt-inverse-frequency weighted RF", **m03_metrics},
])
print("\nOverall held-out comparison — identical test cells:")
display(m01_m02_m03_overall)

comparison_rows = []
for b, label in enumerate(weight_labels):
    m = test_bin == b
    n = int(m.sum())
    if n == 0:
        continue
    yt = test_observed[m]
    p1, p2, p3 = m01_pred[m], m02_pred[m], m03_pred[m]
    row = {
        "observed_bin": label,
        "n_test": n,
        "observed_mean": float(yt.mean()),
    }
    for name, p in [("M01", p1), ("M02", p2), ("M03", p3)]:
        row[f"{name}_pred_mean"] = float(p.mean())
        row[f"{name}_bias"] = float(np.mean(p - yt))
        row[f"{name}_MAE"] = float(mean_absolute_error(yt, p))
        row[f"{name}_RMSE"] = float(mean_squared_error(yt, p) ** 0.5)
        row[f"{name}_Spearman"] = float(spearmanr(yt, p).statistic) if n >= 3 else np.nan
    comparison_rows.append(row)

m01_m02_m03_by_abundance = pd.DataFrame(comparison_rows)
print("\nAbundance-stratified held-out comparison:")
display(m01_m02_m03_by_abundance)

fig, ax = plt.subplots(figsize=(8, 4))
xpos = np.arange(len(m01_m02_m03_by_abundance))
ax.plot(xpos, m01_m02_m03_by_abundance["observed_mean"], marker="o", label="Observed")
ax.plot(xpos, m01_m02_m03_by_abundance["M01_pred_mean"], marker="o", label="M01 unweighted")
ax.plot(xpos, m01_m02_m03_by_abundance["M02_pred_mean"], marker="o", label="M02 inverse-weighted")
ax.plot(xpos, m01_m02_m03_by_abundance["M03_pred_mean"], marker="o", label="M03 sqrt-weighted")
ax.set_xticks(xpos)
ax.set_xticklabels(m01_m02_m03_by_abundance["observed_bin"], rotation=30, ha="right")
ax.set_xlabel("Observed Musanga abundance class")
ax.set_ylabel("Mean Musanga probability")
ax.set_title("M01 vs M02 vs M03 — identical held-out cells")
ax.legend()
plt.tight_layout()
plt.show()

print("\nM03 interpretation:")
print("- M03 is successful if it improves moderate/high-abundance predictions relative to M01")
print("  while producing substantially less low-abundance inflation than M02.")
print("- If the trade-off remains weak, stop tuning embedding weights and move to Sentinel-2 temporal/spectral predictors.")


## BLOCK 08E — M04 multi-year Google Satellite Embeddings (2019–2021)

M04 tests whether **temporal context around the August 2020 UAV survey** improves Musanga prediction. It combines the annual 64-D Google Satellite Embeddings for **2019, 2020 and 2021** (192 predictors). The existing 2020 raster and UAV 10-m target are reused; only missing 2019/2021 embedding rasters are downloaded once and cached in the same Drive output folder. Annual rasters are aligned to the authoritative 2020 grid before modelling.


In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08E — M04 MULTI-YEAR GOOGLE SATELLITE EMBEDDINGS (2019–2021)")
print("=" * 100)

MULTI_YEARS = [2019, 2020, 2021]
MULTI_EMBED_TIFS = {
    y: OUT_DIR / f"Yoko_GoogleSatelliteEmbedding_{y}_10m.tif"
    for y in MULTI_YEARS
}
MULTI_EMBED_TIFS[2020] = EMBED_TIF  # reuse the already cached authoritative 2020 raster

# Download only missing annual embeddings. No UAV aggregation is repeated.
for year in MULTI_YEARS:
    path = MULTI_EMBED_TIFS[year]
    if path.exists():
        print(f"{year}: reusing {path}")
        continue

    print(f"{year}: downloading 64-band annual embedding once...")
    annual_collection = (
        ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .filterBounds(roi)
    )
    n_images = annual_collection.size().getInfo()
    print(f"  intersecting images: {n_images}")
    if n_images == 0:
        raise RuntimeError(f"No Google Satellite Embedding images found for Yoko in {year}.")

    annual_embedding = annual_collection.mosaic().select(band_names)
    geemap.ee_export_image(
        annual_embedding,
        filename=str(path),
        scale=10,
        region=roi,
        file_per_band=False,
    )
    if not path.exists():
        raise RuntimeError(f"Embedding export failed for {year}: {path}")

# 2020 grid is authoritative because the UAV target was aggregated to this grid.
with rasterio.open(EMBED_TIF) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_shape = (ref.height, ref.width)

year_cubes = {}
year_valid = {}

for year in MULTI_YEARS:
    path = MULTI_EMBED_TIFS[year]
    with rasterio.open(path) as src:
        if src.count != 64:
            raise ValueError(f"{year}: expected 64 bands, got {src.count}")

        same_grid = (
            src.crs == ref_crs
            and src.transform.almost_equals(ref_transform)
            and (src.height, src.width) == ref_shape
        )

        if same_grid:
            cube = src.read().astype("float32")
        else:
            print(f"{year}: aligning annual embedding to the 2020 grid...")
            cube = np.full((64, *ref_shape), np.nan, dtype="float32")
            for band_i in range(64):
                reproject(
                    source=rasterio.band(src, band_i + 1),
                    destination=cube[band_i],
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    dst_nodata=np.nan,
                    resampling=Resampling.bilinear,
                    init_dest_nodata=True,
                )

        valid = np.all(np.isfinite(cube), axis=0)
        if same_grid and src.nodata is not None:
            valid &= np.all(cube != src.nodata, axis=0)

    year_cubes[year] = cube
    year_valid[year] = valid
    print(f"{year}: valid cells on 2020 grid = {int(valid.sum()):,}")

multi_mask = good.copy()
for year in MULTI_YEARS:
    multi_mask &= year_valid[year]

multi_rows, multi_cols = np.where(multi_mask)
X_multi = np.concatenate(
    [year_cubes[y][:, multi_mask].T for y in MULTI_YEARS],
    axis=1,
)
target_multi = obs[multi_mask]

# Keep the exact M01 held-out cells whenever all three annual embeddings are valid.
# Cell identity is defined by row/column on the authoritative 2020 grid.
base_flat = rows * ref_shape[1] + cols
multi_flat = multi_rows * ref_shape[1] + multi_cols
multi_lookup = {int(v): i for i, v in enumerate(multi_flat)}
train_multi = np.array([multi_lookup[int(base_flat[i])] for i in train_idx if int(base_flat[i]) in multi_lookup], dtype=int)
test_multi = np.array([multi_lookup[int(base_flat[i])] for i in test_idx if int(base_flat[i]) in multi_lookup], dtype=int)

print("\nM04 data:")
print("Predictors:", X_multi.shape[1], "(64 bands × 3 years)")
print("Valid 3-year cells:", len(target_multi))
print("Training cells retained from M01 split:", len(train_multi), "/", len(train_idx))
print("Test cells retained from M01 split:", len(test_multi), "/", len(test_idx))

M04_MODEL_MODEL_FILE=MODEL_CACHE_DIR/"m04_model.joblib"
M04_MODEL_PRED_FILE=MODEL_CACHE_DIR/"m04_model_heldout.npz"
if (not FORCE_REFIT_MODELS) and M04_MODEL_MODEL_FILE.exists() and M04_MODEL_PRED_FILE.exists():
    print("Reusing cached m04_model model/results")
    m04_model=joblib.load(M04_MODEL_MODEL_FILE)
    _c=np.load(M04_MODEL_PRED_FILE)
    m04_pred=_c["pred"]; m04_obs=_c["obs"]
else:
    m04_model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",n_jobs=-1,random_state=RANDOM_SEED)
    m04_model.fit(X_multi[train_multi],target_multi[train_multi])
    m04_pred=np.clip(m04_model.predict(X_multi[test_multi]),0,1); m04_obs=target_multi[test_multi]
    joblib.dump(m04_model,M04_MODEL_MODEL_FILE)
    np.savez_compressed(M04_MODEL_PRED_FILE,pred=m04_pred,obs=m04_obs)
    print("Cached m04_model model/results")
m04_metrics = regression_metrics(m04_obs, m04_pred)

# Fair M01 comparison on exactly the same M04 test cells.
base_lookup = {int(v): i for i, v in enumerate(base_flat)}
base_test_positions = np.array([base_lookup[int(multi_flat[i])] for i in test_multi], dtype=int)
m01_same_pred = np.clip(best_model.predict(X[base_test_positions]), 0, 1)
m01_same_obs = target[base_test_positions]
m01_same_metrics = regression_metrics(m01_same_obs, m01_same_pred)

m04_overall = pd.DataFrame([
    {"model": "M01 2020 embedding — same M04 test cells", **m01_same_metrics},
    {"model": "M04 embeddings 2019+2020+2021", **m04_metrics},
])
print("\nFair held-out comparison — identical cells:")
display(m04_overall)

# Same abundance diagnostic used for M01–M03.
m04_test_bin = np.digitize(m04_obs, weight_edges[1:-1], right=False)
m04_rows = []
for b, label in enumerate(weight_labels):
    m = m04_test_bin == b
    n = int(m.sum())
    if n == 0:
        continue
    yt = m04_obs[m]
    p1 = m01_same_pred[m]
    p4 = m04_pred[m]
    m04_rows.append({
        "observed_bin": label,
        "n_test": n,
        "observed_mean": float(yt.mean()),
        "M01_pred_mean": float(p1.mean()),
        "M04_pred_mean": float(p4.mean()),
        "M01_bias": float(np.mean(p1 - yt)),
        "M04_bias": float(np.mean(p4 - yt)),
        "M01_MAE": float(mean_absolute_error(yt, p1)),
        "M04_MAE": float(mean_absolute_error(yt, p4)),
        "M01_RMSE": float(mean_squared_error(yt, p1) ** 0.5),
        "M04_RMSE": float(mean_squared_error(yt, p4) ** 0.5),
        "M01_Spearman": float(spearmanr(yt, p1).statistic) if n >= 3 else np.nan,
        "M04_Spearman": float(spearmanr(yt, p4).statistic) if n >= 3 else np.nan,
    })

m04_by_abundance = pd.DataFrame(m04_rows)
print("\nAbundance-stratified M01 vs M04:")
display(m04_by_abundance)

fig, ax = plt.subplots(figsize=(8, 4))
xpos = np.arange(len(m04_by_abundance))
ax.plot(xpos, m04_by_abundance["observed_mean"], marker="o", label="Observed")
ax.plot(xpos, m04_by_abundance["M01_pred_mean"], marker="o", label="M01 2020")
ax.plot(xpos, m04_by_abundance["M04_pred_mean"], marker="o", label="M04 2019+2020+2021")
ax.set_xticks(xpos)
ax.set_xticklabels(m04_by_abundance["observed_bin"], rotation=30, ha="right")
ax.set_xlabel("Observed Musanga abundance class")
ax.set_ylabel("Mean Musanga probability")
ax.set_title("Single-year vs multi-year satellite embeddings")
ax.legend()
plt.tight_layout()
plt.show()

print("\nM04 interpretation:")
print("- A clear improvement over M01 on identical cells means annual temporal context adds Musanga information.")
print("- Pay particular attention to the 0.20–0.50 and >0.50 bins, where M01 strongly underpredicted.")
print("- If M04 remains strongly compressed in the upper tail, the next step should be Sentinel-2 temporal/spectral trajectories rather than further embedding-weight tuning.")


## BLOCK 08F — M05 Sentinel-2 temporal/spectral model (2019–2021)

M05 uses cloud-masked quarterly Sentinel-2 composites for 2019–2021: 10 reflectance bands plus NDVI, NDMI, NBR and red-edge NDVI per quarter (168 predictors). The stack is cached and aligned to the authoritative 2020 UAV/embedding grid.

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08F — M05 SENTINEL-2 TEMPORAL / SPECTRAL MODEL")
print("=" * 100)
S2_TIF = OUT_DIR / "Yoko_Sentinel2_quarterly_features_2019_2021_10m.tif"
S2_BANDS=["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
S2_FEATURES=S2_BANDS+["NDVI","NDMI","NBR","RENDVI"]
def mask_s2(img):
    scl=img.select("SCL")
    clear=scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    r=img.select(S2_BANDS).multiply(0.0001)
    out=r.addBands([
        r.normalizedDifference(["B8","B4"]).rename("NDVI"),
        r.normalizedDifference(["B8","B11"]).rename("NDMI"),
        r.normalizedDifference(["B8","B12"]).rename("NBR"),
        r.normalizedDifference(["B8A","B5"]).rename("RENDVI")
    ]).updateMask(clear)
    return out.copyProperties(img, ["system:time_start", "system:index"])
if not S2_TIF.exists():
    s2=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi)
        .filterDate("2019-01-01","2022-01-01").filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",80)).map(mask_s2))
    print("Total candidate S2 scenes:", s2.size().getInfo())
    ims=[]; used_periods=[]
    for year in [2019,2020,2021]:
        for q,month in enumerate([1,4,7,10],1):
            st=ee.Date.fromYMD(year,month,1); en=st.advance(3,"month")
            qcol=s2.filterDate(st,en)
            n=qcol.size().getInfo()
            print(f"{year} Q{q}: {n} scenes")
            if n == 0:
                print(f"  WARNING: skipping {year} Q{q} because no Sentinel-2 scenes are available.")
                continue
            names=[f"S2_{year}_Q{q}_{b}" for b in S2_FEATURES]
            ims.append(qcol.median().select(S2_FEATURES).rename(names))
            used_periods.append(f"{year}_Q{q}")
    if not ims:
        raise RuntimeError("No Sentinel-2 quarterly composites available over Yoko.")
    stack=ee.Image.cat(ims).toFloat()
    print("Quarterly periods used:", used_periods)
    print("Downloading/caching", len(ims)*len(S2_FEATURES), "Sentinel-2 predictors once...")
    geemap.ee_export_image(stack,filename=str(S2_TIF),scale=10,region=roi,file_per_band=False)
    if not S2_TIF.exists():
        raise RuntimeError("Sentinel-2 export failed; no cache file was created.")
else:
    print("Reusing:",S2_TIF)
if not S2_TIF.exists():
    raise FileNotFoundError(f"Missing Sentinel-2 cache after export attempt: {S2_TIF}")
with rasterio.open(S2_TIF) as src:
    s2_cube=np.full((src.count,*ref_shape),np.nan,dtype="float32")
    same=src.crs==ref_crs and src.transform.almost_equals(ref_transform) and (src.height,src.width)==ref_shape
    if same:
        s2_cube=src.read().astype("float32")
        s2_valid=np.all(np.isfinite(s2_cube),axis=0)
        if src.nodata is not None: s2_valid &= np.all(s2_cube != src.nodata,axis=0)
    else:
        for bi in range(src.count):
            reproject(source=rasterio.band(src,bi+1),destination=s2_cube[bi],src_transform=src.transform,src_crs=src.crs,src_nodata=src.nodata,dst_transform=ref_transform,dst_crs=ref_crs,dst_nodata=np.nan,resampling=Resampling.bilinear,init_dest_nodata=True)
        s2_valid=np.all(np.isfinite(s2_cube),axis=0)
s2_mask=good & s2_valid
sr,sc=np.where(s2_mask); s2_flat=sr*ref_shape[1]+sc
s2_lookup={int(v):i for i,v in enumerate(s2_flat)}
X_s2=s2_cube[:,s2_mask].T; y_s2=obs[s2_mask]
train_s2=np.array([s2_lookup[int(base_flat[i])] for i in train_idx if int(base_flat[i]) in s2_lookup],dtype=int)
test_s2=np.array([s2_lookup[int(base_flat[i])] for i in test_idx if int(base_flat[i]) in s2_lookup],dtype=int)
M05_MODEL_MODEL_FILE=MODEL_CACHE_DIR/"m05_model.joblib"
M05_MODEL_PRED_FILE=MODEL_CACHE_DIR/"m05_model_heldout.npz"
if (not FORCE_REFIT_MODELS) and M05_MODEL_MODEL_FILE.exists() and M05_MODEL_PRED_FILE.exists():
    print("Reusing cached m05_model model/results")
    m05_model=joblib.load(M05_MODEL_MODEL_FILE)
    _c=np.load(M05_MODEL_PRED_FILE)
    m05_pred=_c["pred"]; m05_obs=_c["obs"]
else:
    m05_model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",n_jobs=-1,random_state=RANDOM_SEED)
    m05_model.fit(X_s2[train_s2],y_s2[train_s2])
    m05_pred=np.clip(m05_model.predict(X_s2[test_s2]),0,1); m05_obs=y_s2[test_s2]
    joblib.dump(m05_model,M05_MODEL_MODEL_FILE)
    np.savez_compressed(M05_MODEL_PRED_FILE,pred=m05_pred,obs=m05_obs)
    print("Cached m05_model model/results")
m05_metrics=regression_metrics(m05_obs,m05_pred)
base_lookup={int(v):i for i,v in enumerate(base_flat)}
bp=np.array([base_lookup[int(s2_flat[i])] for i in test_s2],dtype=int)
p01=np.clip(best_model.predict(X[bp]),0,1)
print("M05 predictors:",X_s2.shape[1],"train:",len(train_s2),"test:",len(test_s2))
display(pd.DataFrame([
 {"model":"M01 2020 embedding — same cells",**regression_metrics(m05_obs,p01)},
 {"model":"M05 Sentinel-2 quarterly 2019–2021",**m05_metrics}
]))
b5=np.digitize(m05_obs,weight_edges[1:-1]); out=[]
for b,label in enumerate(weight_labels):
    mm=b5==b
    if mm.any():
        yt=m05_obs[mm]; pp=m05_pred[mm]
        out.append({"observed_bin":label,"n":int(mm.sum()),"observed_mean":float(yt.mean()),"predicted_mean":float(pp.mean()),"bias":float(np.mean(pp-yt)),"RMSE":float(mean_squared_error(yt,pp)**0.5),"Spearman":float(spearmanr(yt,pp).statistic) if mm.sum()>=3 else np.nan})
m05_by_abundance=pd.DataFrame(out); display(m05_by_abundance)

## BLOCK 08G — M06 fusion: multi-year embeddings + Sentinel-2 temporal

M06 concatenates M04's 192 annual embedding predictors with M05's 168 explicit Sentinel-2 temporal/spectral predictors and compares them on identical held-out cells.

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08G — M06 EMBEDDING + SENTINEL-2 FUSION")
print("=" * 100)
fusion_mask=multi_mask & s2_valid
fr,fc=np.where(fusion_mask); fusion_flat=fr*ref_shape[1]+fc
fusion_lookup={int(v):i for i,v in enumerate(fusion_flat)}
Xe=np.concatenate([year_cubes[y][:,fusion_mask].T for y in MULTI_YEARS],axis=1)
Xs=s2_cube[:,fusion_mask].T
X_m06=np.concatenate([Xe,Xs],axis=1); y_m06=obs[fusion_mask]
train_m06=np.array([fusion_lookup[int(base_flat[i])] for i in train_idx if int(base_flat[i]) in fusion_lookup],dtype=int)
test_m06=np.array([fusion_lookup[int(base_flat[i])] for i in test_idx if int(base_flat[i]) in fusion_lookup],dtype=int)
M06_MODEL_MODEL_FILE=MODEL_CACHE_DIR/"m06_model.joblib"
M06_MODEL_PRED_FILE=MODEL_CACHE_DIR/"m06_model_heldout.npz"
if (not FORCE_REFIT_MODELS) and M06_MODEL_MODEL_FILE.exists() and M06_MODEL_PRED_FILE.exists():
    print("Reusing cached m06_model model/results")
    m06_model=joblib.load(M06_MODEL_MODEL_FILE)
    _c=np.load(M06_MODEL_PRED_FILE)
    m06_pred=_c["pred"]; y6=_c["obs"]
else:
    m06_model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",n_jobs=-1,random_state=RANDOM_SEED)
    m06_model.fit(X_m06[train_m06],y_m06[train_m06])
    m06_pred=np.clip(m06_model.predict(X_m06[test_m06]),0,1); y6=y_m06[test_m06]
    joblib.dump(m06_model,M06_MODEL_MODEL_FILE)
    np.savez_compressed(M06_MODEL_PRED_FILE,pred=m06_pred,obs=y6)
    print("Cached m06_model model/results")
m04_lookup={int(v):i for i,v in enumerate(multi_flat)}
p4=np.clip(m04_model.predict(X_multi[np.array([m04_lookup[int(fusion_flat[i])] for i in test_m06])]),0,1)
p5=np.clip(m05_model.predict(X_s2[np.array([s2_lookup[int(fusion_flat[i])] for i in test_m06])]),0,1)
print("M06 predictors:",X_m06.shape[1],"train:",len(train_m06),"test:",len(test_m06))
display(pd.DataFrame([
 {"model":"M04 embeddings 2019–2021",**regression_metrics(y6,p4)},
 {"model":"M05 Sentinel-2 temporal",**regression_metrics(y6,p5)},
 {"model":"M06 embeddings + Sentinel-2",**regression_metrics(y6,m06_pred)}
]))

## BLOCK 08H — M07 PlanetScope spatial/spectral model

M07 uses the Planet NICFI Africa basemap in Earth Engine as the immediately available PlanetScope-scale source. It derives B/G/R/N, NDVI and native-resolution local variability features before export to the 10-m modelling grid. A dedicated ~3-m PlanetScope scene near 24–25 August 2020 can later replace this source without changing the downstream model design.

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08H — M07 PLANET NICFI SPATIAL / SPECTRAL MODEL")
print("=" * 100)
PLANET_TIF=OUT_DIR/"Yoko_PlanetNICFI_2020_spatial_features_10m.tif"
M07_AVAILABLE=True
if not PLANET_TIF.exists():
    try:
        pic=(ee.ImageCollection("projects/planet-nicfi/assets/basemaps/africa")
             .filterBounds(roi).filterDate("2020-06-01","2021-01-01"))
        npic=pic.size().getInfo(); print("Planet NICFI mosaics:",npic)
        if npic==0: raise RuntimeError("No NICFI mosaics found.")
        p=pic.median().select(["B","G","R","N"]).multiply(0.0001)
        nd=p.normalizedDifference(["N","R"]).rename("NDVI")
        base=p.addBands(nd).rename(["B","G","R","N","NDVI"])
        tex=base.reduceNeighborhood(ee.Reducer.stdDev(),ee.Kernel.square(1,"pixels")).rename(["B_std","G_std","R_std","N_std","NDVI_std"])
        pst=base.addBands(tex).toFloat()
        geemap.ee_export_image(pst,filename=str(PLANET_TIF),scale=10,region=roi,file_per_band=False)
    except Exception as e:
        M07_AVAILABLE=False
        print("Planet NICFI unavailable:",repr(e))
        print("If this is a licence/access issue, enable Planet NICFI for this Earth Engine account and rerun.")
if PLANET_TIF.exists():
    with rasterio.open(PLANET_TIF) as src:
        p_cube=np.full((src.count,*ref_shape),np.nan,dtype="float32")
        same=src.crs==ref_crs and src.transform.almost_equals(ref_transform) and (src.height,src.width)==ref_shape
        if same:
            p_cube=src.read().astype("float32"); p_valid=np.all(np.isfinite(p_cube),axis=0)
            if src.nodata is not None: p_valid &= np.all(p_cube != src.nodata,axis=0)
        else:
            for bi in range(src.count):
                reproject(source=rasterio.band(src,bi+1),destination=p_cube[bi],src_transform=src.transform,src_crs=src.crs,src_nodata=src.nodata,dst_transform=ref_transform,dst_crs=ref_crs,dst_nodata=np.nan,resampling=Resampling.bilinear,init_dest_nodata=True)
            p_valid=np.all(np.isfinite(p_cube),axis=0)
    pmask=good&p_valid; pr,pc=np.where(pmask); pflat=pr*ref_shape[1]+pc
    plook={int(v):i for i,v in enumerate(pflat)}
    X_p=p_cube[:,pmask].T; y_p=obs[pmask]
    trp=np.array([plook[int(base_flat[i])] for i in train_idx if int(base_flat[i]) in plook],dtype=int)
    tep=np.array([plook[int(base_flat[i])] for i in test_idx if int(base_flat[i]) in plook],dtype=int)
    M07_MODEL_MODEL_FILE=MODEL_CACHE_DIR/"m07_model.joblib"
M07_MODEL_PRED_FILE=MODEL_CACHE_DIR/"m07_model_heldout.npz"
if (not FORCE_REFIT_MODELS) and M07_MODEL_MODEL_FILE.exists() and M07_MODEL_PRED_FILE.exists():
    print("Reusing cached m07_model model/results")
    m07_model=joblib.load(M07_MODEL_MODEL_FILE)
    _c=np.load(M07_MODEL_PRED_FILE)
    m07_pred=_c["pred"]; y7=_c["obs"]
else:
    m07_model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",n_jobs=-1,random_state=RANDOM_SEED)
    m07_model.fit(X_p[trp],y_p[trp])
    m07_pred=np.clip(m07_model.predict(X_p[tep]),0,1); y7=y_p[tep]
    joblib.dump(m07_model,M07_MODEL_MODEL_FILE)
    np.savez_compressed(M07_MODEL_PRED_FILE,pred=m07_pred,obs=y7)
    print("Cached m07_model model/results")
    bp=np.array([base_lookup[int(pflat[i])] for i in tep],dtype=int); p1=np.clip(best_model.predict(X[bp]),0,1)
    print("M07 predictors:",X_p.shape[1],"train:",len(trp),"test:",len(tep))
    display(pd.DataFrame([
      {"model":"M01 2020 embedding — same cells",**regression_metrics(y7,p1)},
      {"model":"M07 Planet NICFI spatial/spectral",**regression_metrics(y7,m07_pred)}
    ]))
    M07_AVAILABLE=True

## BLOCK 08I — M08 full fusion: embeddings + Sentinel-2 + Planet

M08 combines the three information domains on common cells: annual learned embeddings, explicit Sentinel-2 temporal/spectral predictors, and Planet fine-spatial predictors. M04–M08 are evaluated on exactly the same held-out cells.

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08I — M08 FULL FUSION")
print("=" * 100)
if not M07_AVAILABLE:
    print("M08 skipped: M07 Planet predictors unavailable.")
else:
    fullmask=multi_mask&s2_valid&p_valid
    rr,cc=np.where(fullmask); ff=rr*ref_shape[1]+cc; fl={int(v):i for i,v in enumerate(ff)}
    X08=np.concatenate([
        np.concatenate([year_cubes[y][:,fullmask].T for y in MULTI_YEARS],axis=1),
        s2_cube[:,fullmask].T,p_cube[:,fullmask].T],axis=1)
    y08=obs[fullmask]
    tr=np.array([fl[int(base_flat[i])] for i in train_idx if int(base_flat[i]) in fl],dtype=int)
    te=np.array([fl[int(base_flat[i])] for i in test_idx if int(base_flat[i]) in fl],dtype=int)
    M08_MODEL_MODEL_FILE=MODEL_CACHE_DIR/"m08_model.joblib"
M08_MODEL_PRED_FILE=MODEL_CACHE_DIR/"m08_model_heldout.npz"
if (not FORCE_REFIT_MODELS) and M08_MODEL_MODEL_FILE.exists() and M08_MODEL_PRED_FILE.exists():
    print("Reusing cached m08_model model/results")
    m08_model=joblib.load(M08_MODEL_MODEL_FILE)
    _c=np.load(M08_MODEL_PRED_FILE)
    p8=_c["pred"]; yt=_c["obs"]
else:
    m08_model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",n_jobs=-1,random_state=RANDOM_SEED)
    m08_model.fit(X08[tr],y08[tr])
    p8=np.clip(m08_model.predict(X08[te]),0,1); yt=y08[te]
    joblib.dump(m08_model,M08_MODEL_MODEL_FILE)
    np.savez_compressed(M08_MODEL_PRED_FILE,pred=p8,obs=yt)
    print("Cached m08_model model/results")
    pos4=np.array([m04_lookup[int(ff[i])] for i in te]); pos5=np.array([s2_lookup[int(ff[i])] for i in te])
    pos6=np.array([fusion_lookup[int(ff[i])] for i in te]); pos7=np.array([plook[int(ff[i])] for i in te])
    p4=np.clip(m04_model.predict(X_multi[pos4]),0,1); p5=np.clip(m05_model.predict(X_s2[pos5]),0,1)
    p6=np.clip(m06_model.predict(X_m06[pos6]),0,1); p7=np.clip(m07_model.predict(X_p[pos7]),0,1)
    final_compare=pd.DataFrame([
      {"model":"M04 embeddings 2019–2021",**regression_metrics(yt,p4)},
      {"model":"M05 Sentinel-2 temporal",**regression_metrics(yt,p5)},
      {"model":"M06 embeddings + S2",**regression_metrics(yt,p6)},
      {"model":"M07 Planet spatial/spectral",**regression_metrics(yt,p7)},
      {"model":"M08 embeddings + S2 + Planet",**regression_metrics(yt,p8)}
    ]).sort_values("RMSE")
    print("M08 predictors:",X08.shape[1],"train:",len(tr),"test:",len(te))
    print("FINAL FAIR COMPARISON — identical held-out cells")
    display(final_compare)
    bb=np.digitize(yt,weight_edges[1:-1]); oo=[]
    for b,label in enumerate(weight_labels):
        mm=bb==b
        if mm.any():
            yy=yt[mm]; pp=p8[mm]
            oo.append({"observed_bin":label,"n":int(mm.sum()),"observed_mean":float(yy.mean()),"M08_pred_mean":float(pp.mean()),"bias":float(np.mean(pp-yy)),"RMSE":float(mean_squared_error(yy,pp)**0.5),"Spearman":float(spearmanr(yy,pp).statistic) if mm.sum()>=3 else np.nan})
    m08_by_abundance=pd.DataFrame(oo); display(m08_by_abundance)

## BLOCK 09 — Held-out diagnostic plots

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 09 — HELD-OUT DIAGNOSTICS")
print("=" * 100)

test_pred = np.clip(best_model.predict(X[test_idx]), 0, 1)
test_obs = target[test_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(test_obs, test_pred, s=8, alpha=0.35)
lim = max(float(test_obs.max()), float(test_pred.max()))
axes[0].plot([0, lim], [0, lim], "--")
axes[0].set_xlabel("Observed UAV mean probability")
axes[0].set_ylabel("Predicted probability")
axes[0].set_title(f"Held-out cells — {best_name}")

res = test_obs - test_pred
axes[1].scatter(test_pred, res, s=8, alpha=0.35)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Observed − predicted")
axes[1].set_title("Residuals")
plt.tight_layout()
plt.show()

## BLOCK 10 — Refit on all Yoko cells and export maps

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 10 — REFIT / EXPORT MAPS")
print("=" * 100)

final_model = models[best_name]
final_model.fit(X, target)
joblib.dump(final_model, MODEL_FILE)

pred_all = np.clip(final_model.predict(X), 0, 1).astype("float32")
pred_grid = np.full(dst_shape, -9999.0, dtype="float32")
resid_grid = np.full(dst_shape, -9999.0, dtype="float32")
pred_grid[rows, cols] = pred_all
resid_grid[rows, cols] = target - pred_all

for path, arr in [(PRED_TIF, pred_grid), (RESID_TIF, resid_grid)]:
    with rasterio.open(path, "w", **profile1) as dst:
        dst.write(arr, 1)

summary = {
    "site": "Yoko",
    "embedding_year": EMBEDDING_YEAR,
    "target": "mean UAV-derived Musanga probability on Google embedding grid",
    "threshold_used_for_target": False,
    "n_cells": int(len(target)),
    "test_fraction": TEST_SIZE,
    "split": "random single-site development split",
    "selected_model": best_name,
    "benchmark": results_df.replace({np.nan: None}).to_dict(orient="records"),
    "outputs": {
        "observed_10m": str(OBS_TIF),
        "predicted_10m": str(PRED_TIF),
        "residual_10m": str(RESID_TIF),
        "training_table": str(TABLE_CSV),
        "model": str(MODEL_FILE),
    },
}
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved prediction:", PRED_TIF)
print("Saved residual:", RESID_TIF)
print("Saved summary:", SUMMARY_JSON)

## BLOCK 11 — Interpretation and next step

This notebook is a **Yoko development test**, not yet a test of geographic transferability.

The random held-out metrics test whether the 2020 Google Satellite Embedding contains enough local information to reconstruct variation in the UAV-derived Musanga signal at Yoko. Nearby pixels are spatially related, so these metrics should not be interpreted as independent regional validation.

The stronger next experiment is **cross-site transfer**: train at Yoko and predict an independent site with its own UAV Musanga probability raster.

If the embedding-only experiment is promising, the next benchmark should compare:
- Google Satellite Embedding only;
- explicit Sentinel-2 temporal/spectral features;
- Satellite Embedding + Sentinel-2 temporal features.